# Notebook Setup

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display

# Portfolio Setup

In [2]:
tickers = ['AAPL', 'TSLA', 'MSFT', 'META', 'AMZN', 'GOOGL', 'NVDA']  # Ticker der MAG7
benchmark_world_ticker = 'VT'
benchmark_risk_free_ticker = '^TNX'

start_capital = 100_000  # USD
start_date = "2012-05-18"  # gemeinsamer Startpunkt ab Meta-IPO
end_date = "2026-04-01"

confidence_levels = {
    "95 %": 0.05,
    "99 %": 0.01,
}
reference_alpha = confidence_levels["95 %"]

horizons = {
    "1 Jahr": 252,
    "5 Jahre": 1260,
    "10 Jahre": 2520,
    "20 Jahre": 5040,
}

monte_carlo_simulations = 10_000
bootstrap_simulations = 10_000
monte_carlo_seed = 2
bootstrap_seed = 1

In [3]:
data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close']

In [4]:
benchmark_world_data = yf.download(
    benchmark_world_ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close']  # VT (Vanguard Total World Stock ETF)

benchmark_risk_free_data = yf.download(
    benchmark_risk_free_ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close']  # ^TNX (10-Year Treasury Note Yield)

In [5]:
dotcom_data = yf.download(
    '^NDX',  # Nasdaq-ETF als Proxy für Dotcom-Blase
    start="2000-04-01",
    end="2002-10-31",
    auto_adjust=False,
    progress=False,
)['Adj Close'] 

dotcom_log_returns = np.log(dotcom_data / dotcom_data.shift(1)).dropna().values

In [6]:
#Berechnung der täglichen Renditen (diskret und logarithmisch)
returns_discrete = data.pct_change().dropna() #berechnet die prozentuale Veränderung der Renditen - diskret
returns_log = np.log(data / data.shift(1)).dropna() #berechnet die logarithmische Rendite

#Portfolio-Gewichtung
weights = np.array([1/len(tickers)] * len(tickers)) #1/7 Gewichtung für jedes Asset

#Berechnung der Portfolio-Renditen
portfolio_returns_discrete = returns_discrete.dot(weights) #Berechnung der Portfolio-Renditen
portfolio_returns_log = returns_log.dot(weights)

Wir berechnen sowohl **diskrete Renditen** als auch **Log-Renditen**.

- **Diskrete Renditen** sind intuitiv für Prozentveränderungen von einem Tag auf den nächsten.
- **Log-Renditen** sind über die Zeit additiv und deshalb besonders nützlich für Mehrtageshorizonte, historische Aggregation und Monte-Carlo-Simulation.

Das Portfolio wird als **gleichgewichtet** modelliert, also mit `1/7` pro Aktie.

In [7]:
portfolio_returns_discrete.tail()

Date
2026-03-25    0.007628
2026-03-26   -0.031970
2026-03-27   -0.027635
2026-03-30   -0.001346
2026-03-31    0.045284
dtype: float64

# Erstellung der Funktionen

## Berechnung der 1-Tages-Risikomodelle

Value at Risk und Expected Shortfall für historische, gaußsche und lognormale Modellierung.

Wir berichten **VaR** und **Expected Shortfall** als **Verlustgrößen**.

- **Positiver Wert:** potenzieller Verlust gegenüber dem Anfangskapital.
- **Negativer Wert:** selbst im betrachteten Quantil liegt der Endwert noch über dem Anfangskapital.

Für die Normalmethode basiert der VaR auf dem linken Renditequantil der geschätzten Verteilung.

In [8]:
# 1. Historisches Risiko (VaR & ES)
def calculate_historical_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()

    var_pct = np.percentile(clean_returns, alpha * 100) # Berechnung des VaR als %-Quantil der Renditen
    tail_returns = clean_returns[clean_returns <= var_pct] #
    es_pct = np.mean(tail_returns)

    return capital * var_pct, capital * es_pct


# 2. Gaußsches Risiko (VaR & ES)
def calculate_gaussian_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()

    mu = np.mean(clean_returns) #Erwartungswert der Renditen
    sigma = np.std(clean_returns) #Volatilität der Renditen
    z_score = stats.norm.ppf(alpha) #z-Score für das Quantil, das dem VaR-Level entspricht

    var_pct = mu + (z_score * sigma) 
    es_pct = mu - sigma * (stats.norm.pdf(z_score) / alpha)

    return capital * var_pct, capital * es_pct


# 3. Lognormales Risiko (VaR & ES)
def calculate_lognormal_risk_1d(log_returns, capital, var_level):
    alpha = var_level
    clean_returns = log_returns.dropna()

    mu_log = np.mean(clean_returns)
    sigma_log = np.std(clean_returns)
    z_score = stats.norm.ppf(alpha)

    log_worst_case = mu_log + (z_score * sigma_log) #Renditen im logarithmischen Raum
    var_pct = np.exp(log_worst_case) - 1 #Umrechnung zurück in diskrete Renditen

    expected_value_lognorm = np.exp(mu_log + (sigma_log**2) / 2) #Erwartungswert der lognormalverteilten Renditen
    tail_factor = stats.norm.cdf(z_score - sigma_log) / alpha #Wahrscheinlichkeit, dass die Renditen im logarithmischen Raum unter dem Worst-Case liegen
    es_pct = (expected_value_lognorm * tail_factor) - 1

    return capital * var_pct, capital * es_pct

## Skalierung auf Zeiträume: 1 Jahr, 5 Jahre und 10 Jahre

- **Historisch:** Für 1 Jahr nutzen wir rollierende historische Fenster. Für 5 und 10 Jahre nutzen wir historisches Bootstrapping, weil es für direkte langfristige Quantile zu wenige echte nicht überlappende Beobachtungen gibt.
- **Gaußsch:** Die Methode kann für alle Horizonte gerechnet werden. Für lange Horizonte wird sie aber annahmestärker, weil Mittelwert linear mit der Zeit und Volatilität mit der Wurzel aus der Zeit skaliert wird.
- **Lognormal / Monte Carlo:** Die Methode erzeugt viele Zukunftspfade auf Basis normalverteilter Log-Renditen. Das ist für längere Horizonte oft anschaulicher als eine reine Skalierung.

In [9]:
def calculate_historical_risk(log_returns, capital, var_level, days):
    if days == 1:
        rolling_discrete = np.exp(log_returns.dropna()) - 1 #keine Aggregation, da nur 1 Tag betrachtet wird
    else:
        rolling_log = log_returns.rolling(window=days).sum().dropna() # Rolling Window und Aggregation

        if len(rolling_log) < 100: #Sicherheitscheck: nicht weniger als 100 Beobachtungen für die Berechnung
            return np.nan, np.nan

        rolling_discrete = np.exp(rolling_log) - 1 #Log zu diskret

    var_pct = np.percentile(rolling_discrete, var_level * 100) 
    tail_returns = rolling_discrete[rolling_discrete <= var_pct]
    es_pct = np.mean(tail_returns)

    return capital * var_pct, capital * es_pct

In [10]:
def calculate_gaussian_risk(returns, capital, var_level, days):
    clean_returns = returns.dropna()

    mu_1d = np.mean(clean_returns) #Erwartungswert der täglichen Renditen
    sigma_1d = np.std(clean_returns) #Volatilität der täglichen Renditen

    mu_nd = mu_1d * days 
    sigma_nd = sigma_1d * np.sqrt(days) #Skalierung der Volatilität mit der Quadratwurzel der Zeit

    z_score = stats.norm.ppf(var_level)

    var_pct = mu_nd + (z_score * sigma_nd)
    es_pct = mu_nd - sigma_nd * (stats.norm.pdf(z_score) / var_level)

    return capital * var_pct, capital * es_pct

In [11]:
def calculate_lognormal_risk(log_returns, capital, var_level, days):
    clean_returns = log_returns.dropna()

    mu_log_1d = np.mean(clean_returns)
    sigma_log_1d = np.std(clean_returns)

    mu_log_nd = mu_log_1d * days
    sigma_log_nd = sigma_log_1d * np.sqrt(days)

    z_score = stats.norm.ppf(var_level)

    log_worst_case = mu_log_nd + (z_score * sigma_log_nd)
    var_pct = np.exp(log_worst_case) - 1

    expected_value_lognorm = np.exp(mu_log_nd + (sigma_log_nd**2) / 2)
    tail_factor = stats.norm.cdf(z_score - sigma_log_nd) / var_level
    es_pct = (expected_value_lognorm * tail_factor) - 1

    return capital * var_pct, capital * es_pct

Monte-Carlo-Simulation wird hier für längere Horizonte genutzt, weil sie viele mögliche Zukunftspfade erzeugt und den Zinseszinseffekt über die Zeit sauber in den Endwerten abbildet. Die Schwäche liegt in der Modellannahme: Verteilung, Mittelwert und Volatilität werden aus der Vergangenheit geschätzt und in die Zukunft fortgeschrieben.

In [12]:
def calculate_monte_carlo_risk(log_returns, capital, var_level, days, simulations=10000, black_swan=False, crash_data=None, monte_carlo_seed=monte_carlo_seed):
    
    clean_returns = log_returns.dropna()

    mu_log_1d = np.mean(clean_returns)
    sigma_log_1d = np.std(clean_returns)

    np.random.seed(monte_carlo_seed) #Setzen des Zufallsgenerators für Reproduzierbarkeit

    simulated_daily_returns = np.random.normal(mu_log_1d, sigma_log_1d, (days, simulations)) #Monte-Carlo Simulation der täglichen log Renditen

    if black_swan and crash_data is not None:
        crash_data_flat = np.asarray(crash_data).flatten() #Erzeugung eines Arrays
        crash_length = len(crash_data_flat)
        
        if days < crash_length:
            # Laufzeit ist kürzer als der Crash: Wir schneiden ein zufälliges Stück aus dem Crash heraus und fügen es in die Simulation ein
            slice_starts = np.random.randint(0, crash_length - days + 1, size=simulations)
            for s in range(simulations):
                start_idx_crash = slice_starts[s]
                simulated_daily_returns[:, s] = crash_data_flat[start_idx_crash : start_idx_crash + days]
        else:
            # Laufzeit ist länger als der Crash
            # Platzierung des Crashs zufällig innerhalb der Simulationsperiode (so dass er nicht immer am Anfang oder Ende liegt)
            crash_start_days = np.random.randint(0, max(1, days - int(crash_length/2)), size=simulations) #Random Starttag des Crashs 
            for s in range(simulations): #s iteriert über alle Simulationen, aktuelles s = Simulation
                start_idx = crash_start_days[s] #Startindex des Crashs in der Iteration s
                end_idx = min(days, start_idx + crash_length) #Endindex des Crashs, abhängig von der Länge der Simulationsperiode
                actual_crash_len = end_idx - start_idx #Tatsächliche Länge des Crashs, die in die Simulation eingefügt wird (kann kürzer sein als der gesamte Crash, wenn er am Ende der Simulationsperiode liegt)
                simulated_daily_returns[start_idx:end_idx, s] = crash_data_flat[:actual_crash_len] #Einfügen der Crash-Daten in die Simulation, Überschreibt die alten Daten 

            
    cumulative_log_returns = np.cumsum(simulated_daily_returns, axis=0)
    portfolio_paths = capital * np.exp(cumulative_log_returns) #
    
    final_values = portfolio_paths[-1, :]

    var_end_value = np.percentile(final_values, var_level * 100)
    tail_values = final_values[final_values <= var_end_value]
    es_end_value = np.mean(tail_values) if len(tail_values) > 0 else var_end_value

    var_PnL = var_end_value - capital #PnL = Profit and Loss, hier berechnet als Endwert - Anfangskapital, vorher var_pct * capital, aber hier mit aboluten Werten 
    es_PnL = es_end_value - capital

    return var_PnL, es_PnL, final_values, portfolio_paths

Bootstrapping simuliert zukünftige Portfoliowerte durch Ziehen mit Zurücklegen aus der historischen Renditeverteilung. Das ist für unser Projekt die pragmatische historische Langfristvariante. Die Stärke ist die Nähe zu beobachteten Renditen. Die Schwäche ist, dass neue Krisenmuster, Regimewechsel oder Zeitabhängigkeiten nicht explizit modelliert werden.

In [13]:
def calculate_bootstrap_risk(log_returns, capital, var_level, days, simulations=10000):
    clean_returns = log_returns.dropna()

    np.random.seed(bootstrap_seed)
    simulated_daily_returns = np.random.choice(clean_returns, size=(days, simulations), replace=True) #Bootstrap-Sampling der täglichen log Renditen
    cumulative_log_returns = np.sum(simulated_daily_returns, axis=0) 
    final_values = capital * np.exp(cumulative_log_returns)

    var_end_value = np.percentile(final_values, var_level * 100)
    tail_values = final_values[final_values <= var_end_value]
    es_end_value = np.mean(tail_values)

    var_PnL = var_end_value - capital
    es_PnL = es_end_value - capital

    return var_PnL, es_PnL

# Berechnung und Ausführung der Funktionen

## 1-Tages-Modelle

In [14]:
# 1-Tages-Blick als Kurzfrist-Referenz (95 %)
one_day_rows = []

hist_var, hist_es = calculate_historical_risk_1d(portfolio_returns_discrete, start_capital, reference_alpha)
one_day_rows.append({'Methode': 'Historisch', 'VaR ($)': hist_var, 'ES ($)': hist_es})

gauss_var, gauss_es = calculate_gaussian_risk_1d(portfolio_returns_discrete, start_capital, reference_alpha)
one_day_rows.append({'Methode': 'Gaußsch', 'VaR ($)': gauss_var, 'ES ($)': gauss_es})

lognorm_var, lognorm_es = calculate_lognormal_risk_1d(portfolio_returns_log, start_capital, reference_alpha)
one_day_rows.append({'Methode': 'Lognormal', 'VaR ($)': lognorm_var, 'ES ($)': lognorm_es})

one_day_results = pd.DataFrame(one_day_rows)
one_day_results['VaR (%)'] = one_day_results['VaR ($)'] / start_capital * 100
one_day_results['ES (%)'] = one_day_results['ES ($)'] / start_capital * 100

display(one_day_results.round(2))

,Methode,VaR ($),ES ($),VaR (%),ES (%)
0,Historisch,-2684.28,-3936.03,-2.68,-3.94
1,Gaußsch,-2640.08,-3345.08,-2.64,-3.35
2,Lognormal,-2634.08,-3316.34,-2.63,-3.32


## Skalierung auf 1 Jahr, 5 und 10 Jahre bei 95 % und 99 %

Die Kernergebnisse werden kompakt in einer Tabelle berichtet.

- **VaR** und **ES** werden als Verlustgrößen gezeigt.
- **Positive Werte** bedeuten Verlust.
- **Negative Werte** bedeuten, dass selbst das betrachtete Quantil noch über dem Anfangskapital liegt.

In [15]:
core_rows = []

for horizon_label, days in horizons.items():
    for confidence_label, alpha_level in confidence_levels.items():
        
        # --- 1. Historisch / Bootstrapping ---
        if days == 252:
            hist_label = 'Historisch (BHS)'
            h_var, h_es = calculate_historical_risk(portfolio_returns_log, start_capital, alpha_level, days)
        else:
            hist_label = 'Historisch (Bootstrapping)'
            h_var, h_es = calculate_bootstrap_risk(
                portfolio_returns_log,
                start_capital,
                alpha_level,
                days,
                simulations=bootstrap_simulations,
            )

        # --- 2. Gaußsch ---
        g_var, g_es = calculate_gaussian_risk(portfolio_returns_discrete, start_capital, alpha_level, days)
        
        # Normal-Szenario
        mc_var_norm, mc_es_norm, _, _ = calculate_monte_carlo_risk(
            portfolio_returns_log, 
            start_capital, 
            alpha_level, 
            days, 
            simulations=monte_carlo_simulations, 
            black_swan=False
        )
        
        # DotCom-Crash-Szenario
        mc_var_swan, mc_es_swan, _, _ = calculate_monte_carlo_risk(
            portfolio_returns_log, 
            start_capital, 
            alpha_level, 
            days, 
            simulations=monte_carlo_simulations, 
            black_swan=True, 
            crash_data=dotcom_log_returns
        )

        # --- Werte in die Tabelle (Dictionary) laden ---
        core_rows.extend([
            {
                'Horizont': horizon_label,
                'Handelstage': days,
                'Konfidenzniveau': confidence_label,
                'Methode': hist_label,
                'VaR ($)': h_var,
                'ES ($)': h_es,
            },
            {
                'Horizont': horizon_label,
                'Handelstage': days,
                'Konfidenzniveau': confidence_label,
                'Methode': 'Gaußsch',
                'VaR ($)': g_var,
                'ES ($)': g_es,
            },
            {
                'Horizont': horizon_label,
                'Handelstage': days,
                'Konfidenzniveau': confidence_label,
                'Methode': 'Lognormal (Normal)',
                'VaR ($)': mc_var_norm,
                'ES ($)': mc_es_norm,
            },
            {
                'Horizont': horizon_label,
                'Handelstage': days,
                'Konfidenzniveau': confidence_label,
                'Methode': 'Lognormal (DotCom-Crash)',
                'VaR ($)': mc_var_swan,
                'ES ($)': mc_es_swan,
            },
        ])

# --- DataFrame formatieren und ausgeben ---
core_results = pd.DataFrame(core_rows)

# Prozentwerte berechnen
core_results['VaR (%)'] = core_results['VaR ($)'] / start_capital * 100
core_results['ES (%)'] = core_results['ES ($)'] / start_capital * 100

# Rundung für die saubere Darstellung
core_results_display = core_results.copy()
core_results_display[['VaR ($)', 'ES ($)', 'VaR (%)', 'ES (%)']] = core_results_display[['VaR ($)', 'ES ($)', 'VaR (%)', 'ES (%)']].round(2)

# Tabelle anzeigen
display(core_results_display)

,Horizont,Handelstage,Konfidenzniveau,Methode,VaR ($),ES ($),VaR (%),ES (%)
0,1 Jahr,252,95 %,Historisch (BHS),-17967.07,-31261.91,-17.97,-31.26
1,1 Jahr,252,95 %,Gaußsch,-10013.00,-21204.58,-10.01,-21.20
2,1 Jahr,252,95 %,Lognormal (Normal),-15482.02,-24595.07,-15.48,-24.60
3,1 Jahr,252,95 %,Lognormal (DotCom-Crash),-64502.40,-66957.00,-64.50,-66.96
4,1 Jahr,252,99 %,Historisch (BHS),-42907.10,-45950.50,-42.91,-45.95
5,1 Jahr,252,99 %,Gaußsch,-28265.55,-37341.45,-28.27,-37.34
6,1 Jahr,252,99 %,Lognormal (Normal),-30802.69,-36496.44,-30.80,-36.50
7,1 Jahr,252,99 %,Lognormal (DotCom-Crash),-68794.78,-69319.33,-68.79,-69.32
8,5 Jahre,1260,95 %,Historisch (Bootstrapping),40546.91,12118.79,40.55,12.12
9,5 Jahre,1260,95 %,Gaußsch,71698.20,46673.07,71.70,46.67


## Robustheitsindikator

Der Robustheitsindikator vergleicht die **VaR-Werte der drei Methoden** je Horizont und Konfidenzniveau.

Berechnet werden:
- **Absolute Spannweite** = größter VaR minus kleinster VaR
- **Relative Spannweite** = absolute Spannweite geteilt durch den Median der VaR-Werte

Die Einteilung **niedrig / mittel / hoch** ist hier eine **didaktische Heuristik**. Sie ist kein statistischer Test. Sie zeigt nur, wie stark die Methoden im Ergebnis auseinanderliegen.

Wichtig: Liegt der Median der VaR-Werte nahe null, kann die relative Spannweite sehr groß werden. Das ist dann ein Hinweis auf hohe Modellabhängigkeit rund um die Gewinn-/Verlustgrenze und kein Beweis für einen Rechenfehler.

In [16]:
robustness_rows = []
core_results_base = core_results[core_results['Methode'] != 'Lognormal (DotCom-Crash)']

for (horizon_label, days, confidence_label), subset in core_results_base.groupby(['Horizont', 'Handelstage', 'Konfidenzniveau']):
    var_values = subset['VaR ($)'].to_numpy(dtype=float)
    absolute_range = float(var_values.max() - var_values.min())
    median_var = float(np.median(var_values))
    denominator = max(abs(median_var), 1e-9)
    relative_range = absolute_range / denominator

    if relative_range < 0.15:
        robustness_class = 'niedrig'
    elif relative_range <= 0.35:
        robustness_class = 'mittel'
    else:
        robustness_class = 'hoch'

    robustness_rows.append({
        'Horizont': horizon_label,
        'Handelstage': days,
        'Konfidenzniveau': confidence_label,
        'Absolute Spannweite VaR ($)': absolute_range,
        'Relative Spannweite VaR': relative_range,
        'Relative Spannweite VaR (%)': relative_range * 100,
        'Robustheitsklasse': robustness_class,
    })

robustness_summary = pd.DataFrame(robustness_rows).sort_values(['Handelstage', 'Konfidenzniveau']).reset_index(drop=True)
robustness_summary[['Absolute Spannweite VaR ($)', 'Relative Spannweite VaR', 'Relative Spannweite VaR (%)']] = robustness_summary[['Absolute Spannweite VaR ($)', 'Relative Spannweite VaR', 'Relative Spannweite VaR (%)']].round(2)

display(robustness_summary)

,Horizont,Handelstage,Konfidenzniveau,Absolute Spannweite VaR ($),Relative Spannweite VaR,Relative Spannweite VaR (%),Robustheitsklasse
0,1 Jahr,252,95 %,7954.08,0.51,51.38,hoch
1,1 Jahr,252,99 %,14641.55,0.48,47.53,hoch
2,5 Jahre,1260,95 %,31151.29,0.76,75.58,hoch
3,5 Jahre,1260,99 %,39469.35,5.12,512.49,hoch
4,10 Jahre,2520,95 %,58355.59,0.23,22.66,mittel
5,10 Jahre,2520,99 %,41661.72,0.39,39.34,hoch
6,20 Jahre,5040,95 %,2399327.49,0.85,84.94,hoch
7,20 Jahre,5040,99 %,861187.66,0.68,68.44,hoch


## Plausibilitätschecks und kurze Hinweise

Dieser Block prüft einige Grundrelationen der Ergebnisse.

Geprüft werden:
- **ES ≥ VaR** je Methode, Horizont und Konfidenzniveau
- **99 % konservativer als 95 %** je Methode und Horizont
- **Negative VaR-Werte** als Hinweis, dass selbst das betrachtete Quantil noch über dem Anfangskapital liegt

Die Hinweise sind bewusst knapp und regelbasiert. Sie ersetzen keine fachliche Diskussion, helfen aber beim schnellen Einordnen auffälliger Fälle.

In [17]:
plausibility_rows = []
issue_rows = []

# 1) ES sollte auf Verlustbasis mindestens so groß wie VaR sein
for _, row in core_results.iterrows():
    passed = bool(row['ES ($)'] >= row['VaR ($)'])
    plausibility_rows.append({
        'Check': 'ES ≥ VaR',
        'Horizont': row['Horizont'],
        'Konfidenzniveau': row['Konfidenzniveau'],
        'Methode': row['Methode'],
        'Ergebnis': 'OK' if passed else 'Prüfen',
    })
    if not passed:
        issue_rows.append({
            'Typ': 'Plausibilitätswarnung',
            'Horizont': row['Horizont'],
            'Konfidenzniveau': row['Konfidenzniveau'],
            'Methode': row['Methode'],
            'Hinweis': 'Expected Shortfall liegt unter dem VaR. Die Vorzeichen- oder Tail-Logik sollte geprüft werden.',
        })

# 2) 99 % sollte konservativer sein als 95 %
for horizon_label in horizons.keys():
    subset = core_results[core_results['Horizont'] == horizon_label]
    for method in subset['Methode'].unique():
        method_subset = subset[subset['Methode'] == method].set_index('Konfidenzniveau')
        if {'95 %', '99 %'}.issubset(method_subset.index):
            var95 = float(method_subset.loc['95 %', 'VaR ($)'])
            var99 = float(method_subset.loc['99 %', 'VaR ($)'])
            es95 = float(method_subset.loc['95 %', 'ES ($)'])
            es99 = float(method_subset.loc['99 %', 'ES ($)'])

            var_passed = bool(var99 >= var95)
            es_passed = bool(es99 >= es95)

            plausibility_rows.append({
                'Check': '99 % VaR ≥ 95 % VaR',
                'Horizont': horizon_label,
                'Konfidenzniveau': '95 % vs. 99 %',
                'Methode': method,
                'Ergebnis': 'OK' if var_passed else 'Prüfen',
            })
            plausibility_rows.append({
                'Check': '99 % ES ≥ 95 % ES',
                'Horizont': horizon_label,
                'Konfidenzniveau': '95 % vs. 99 %',
                'Methode': method,
                'Ergebnis': 'OK' if es_passed else 'Prüfen',
            })

            if not var_passed:
                issue_rows.append({
                    'Typ': 'Plausibilitätswarnung',
                    'Horizont': horizon_label,
                    'Konfidenzniveau': '95 % vs. 99 %',
                    'Methode': method,
                    'Hinweis': 'Der 99-%-VaR ist nicht konservativer als der 95-%-VaR. Das Ergebnis sollte geprüft werden.',
                })
            if not es_passed:
                issue_rows.append({
                    'Typ': 'Plausibilitätswarnung',
                    'Horizont': horizon_label,
                    'Konfidenzniveau': '95 % vs. 99 %',
                    'Methode': method,
                    'Hinweis': 'Der 99-%-Expected-Shortfall ist nicht konservativer als der 95-%-Expected-Shortfall. Das Ergebnis sollte geprüft werden.',
                })

# 3) Negative VaR-Werte als kurze Ergebnis-Hinweise markieren
negative_var_rows = core_results[core_results['VaR ($)'] < 0]
for _, row in negative_var_rows.iterrows():
    issue_rows.append({
        'Typ': 'Ergebnishinweis',
        'Horizont': row['Horizont'],
        'Konfidenzniveau': row['Konfidenzniveau'],
        'Methode': row['Methode'],
        'Hinweis': 'Der VaR ist negativ. Selbst das betrachtete Quantil liegt noch über dem Anfangskapital.',
    })

# 4) Hohe Robustheitsspannweiten knapp markieren
for _, row in robustness_summary.iterrows():
    if row['Robustheitsklasse'] == 'hoch':
        issue_rows.append({
            'Typ': 'Modellhinweis',
            'Horizont': row['Horizont'],
            'Konfidenzniveau': row['Konfidenzniveau'],
            'Methode': 'Methodenvergleich',
            'Hinweis': 'Die Methoden weichen stark voneinander ab. Die Risikoschätzung ist hier besonders modellabhängig.',
        })

plausibility_df = pd.DataFrame(plausibility_rows)
plausibility_summary = (
    plausibility_df
    .groupby('Check', as_index=False)
    .agg(Anzahl=('Ergebnis', 'size'), OK=('Ergebnis', lambda s: int((s == 'OK').sum())), Prüfen=('Ergebnis', lambda s: int((s == 'Prüfen').sum())))
)

issue_df = pd.DataFrame(issue_rows)
if not issue_df.empty:
    issue_df = issue_df.drop_duplicates().sort_values(['Typ', 'Horizont', 'Konfidenzniveau', 'Methode']).reset_index(drop=True)

print('Plausibilitätsübersicht')
display(plausibility_summary)

if not issue_df.empty:
    print('Auffällige Punkte und kurze Hinweise')
    display(issue_df)
else:
    print('Keine Auffälligkeiten in den definierten Plausibilitätschecks.')

Plausibilitätsübersicht


,Check,Anzahl,OK,Prüfen
0,99 % ES ≥ 95 % ES,16,0,16
1,99 % VaR ≥ 95 % VaR,16,0,16
2,ES ≥ VaR,32,0,32


Auffällige Punkte und kurze Hinweise


,Typ,Horizont,Konfidenzniveau,Methode,Hinweis
0,Ergebnishinweis,1 Jahr,95 %,Gaußsch,Der VaR ist negativ. Selbst das betrachtete Qu...
1,Ergebnishinweis,1 Jahr,95 %,Historisch (BHS),Der VaR ist negativ. Selbst das betrachtete Qu...
2,Ergebnishinweis,1 Jahr,95 %,Lognormal (DotCom-Crash),Der VaR ist negativ. Selbst das betrachtete Qu...
3,Ergebnishinweis,1 Jahr,95 %,Lognormal (Normal),Der VaR ist negativ. Selbst das betrachtete Qu...
4,Ergebnishinweis,1 Jahr,99 %,Gaußsch,Der VaR ist negativ. Selbst das betrachtete Qu...
...,...,...,...,...,...
80,Plausibilitätswarnung,5 Jahre,95 % vs. 99 %,Lognormal (Normal),Der 99-%-Expected-Shortfall ist nicht konserva...
81,Plausibilitätswarnung,5 Jahre,99 %,Gaußsch,Expected Shortfall liegt unter dem VaR. Die Vo...
82,Plausibilitätswarnung,5 Jahre,99 %,Historisch (Bootstrapping),Expected Shortfall liegt unter dem VaR. Die Vo...
83,Plausibilitätswarnung,5 Jahre,99 %,Lognormal (DotCom-Crash),Expected Shortfall liegt unter dem VaR. Die Vo...


## Performance-KPIs (optionaler Zusatzblock)

In [18]:
def calculate_performance_kpis(portfolio_returns, benchmark_world_data, benchmark_risk_free_data):
    
    # Aufbereitung der Daten für die Berechnung der KPIs 
    benchmark_returns = benchmark_world_data.pct_change().dropna().squeeze() #.squeeze() "quetscht" die DataFrame-Struktur, damit wir eine Series haben
    risk_free_daily = (benchmark_risk_free_data.dropna() / 100 / 252).squeeze() #/100 um auf Prozente zu kommen 

    # Daten synchronisieren (gleiche Handelstage usw. hier als für flexibilität bei in USA gehandelten Assets gleich)
    aligned_kpi_data = pd.concat([
        portfolio_returns.squeeze().rename('Portfolio'), 
        benchmark_returns.rename('Market'), 
        risk_free_daily.rename('RiskFree')
    ], axis=1, sort=False).dropna()

    port_ret = aligned_kpi_data['Portfolio'] #Portfolio-Renditen (hier Returns, also ret)
    mkt_ret = aligned_kpi_data['Market']
    rf_ret = aligned_kpi_data['RiskFree']

    #Berechnung der KPIs (Tagesbasis und Annualisiert)

    # Beta berechnen (Kovarianz Portfolio&Markt / Varianz Markt)
    cov_matrix = np.cov(port_ret, mkt_ret)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] #Beta als Maß wie stark Portfolio im Vergleich zum Markt schwankt 

    # Erwartungswerte (Durchschnitte)
    mu_rp_daily = np.mean(port_ret) #Erwartungswert der Portfolio-Renditen
    mu_rm_daily = np.mean(mkt_ret) #Erwartungswert der Markt-Renditen
    mu_rf_daily = np.mean(rf_ret) #Erwartungswert der risikofreien Renditen
    sigma_rp_daily = np.std(port_ret) #Standardabweichung der Portfolio-Renditen

    # Sharpe Ratio
    sharpe_daily = (mu_rp_daily - mu_rf_daily) / sigma_rp_daily
    sharpe_ann = sharpe_daily * np.sqrt(252)

    # Roy's Safety First Ratio (Nutzung vermeitlich Risikofreier Zins, fraglich ob USA-Staatsanleihen als risikofreier Zins gilt)
    rsf_daily = (mu_rp_daily - mu_rm_daily) / sigma_rp_daily
    rsf_ann = rsf_daily * np.sqrt(252)

    # Treynor Ratio (direkt Jahr dan Beta annualisiert ist)
    mu_rp_anno = mu_rp_daily * 252
    mu_rf_anno = mu_rf_daily * 252
    treynor_ann = (mu_rp_anno - mu_rf_anno) / beta

    # --- 3. Ausgabe ---
    print(f"\n{' PERFORMANCE KPIS (Annualisiert) ':=^75}")
    print(f"Portfolio Beta:      {beta:.2f}")
    print(f"Sharpe Ratio:        {sharpe_ann:.2f}")
    print(f"Roy's Safety First:  {rsf_ann:.2f}")
    print(f"Treynor Ratio:       {treynor_ann:.4f}")
    
    # Rückgabe der berechneten Werte als Dictionary für die spätere Weiterverwendung
    return {
        "Beta": beta,
        "Sharpe_Ratio": sharpe_ann,
        "Roys_Safety_First": rsf_ann,
        "Treynor_Ratio": treynor_ann
    }


## Performance-KPIs: Ausführung

In [19]:
kpi_results = calculate_performance_kpis(
    portfolio_returns=portfolio_returns_discrete, 
    benchmark_world_data=benchmark_world_data, 
    benchmark_risk_free_data=benchmark_risk_free_data
)


===================== PERFORMANCE KPIS (Annualisiert) =====================
Portfolio Beta:      1.28
Sharpe Ratio:        1.16
Roy's Safety First:  0.81
Treynor Ratio:       0.2425


Kurze Interpretation der KPIs:

- **Beta:** Das Portfolio schwankt stärker als der Weltmarkt. Ein Beta über 1 bedeutet höhere Marktsensitivität.
- **Sharpe Ratio:** Misst die Überrendite pro Einheit Gesamtrisiko. Ein höherer Wert ist besser.
- **Roy’s Safety First:** Vergleicht die Portfoliorendite mit einer Mindest- bzw. Vergleichsrendite.
- **Treynor Ratio:** Misst die Überrendite pro Einheit systematischen Risikos (Beta).

# Visualisierungen 

In [20]:
import plotly.graph_objects as go

def plot_monte_carlo_fan_chart(portfolio_paths, start_capital, var_level=0.05, title="Monte Carlo Fan-Chart"):
    """
    Erstellt ein interaktives Plotly Fan-Chart mit dynamischen Quantilen basierend auf dem VaR-Level.
    """
    days = portfolio_paths.shape[0]
    
    # 1. Dynamische Berechnung der Grenzen
    # var_level ist z.B. 0.05 -> lower_pct = 5, upper_pct = 95
    # var_level ist z.B. 0.20 -> lower_pct = 20, upper_pct = 80
    lower_pct = var_level * 100
    upper_pct = (1 - var_level) * 100
    
    # Perzentile und Median berechnen
    median_path = np.median(portfolio_paths, axis=1)
    lower_bound = np.percentile(portfolio_paths, lower_pct, axis=1)
    upper_bound = np.percentile(portfolio_paths, upper_pct, axis=1)

    # Tag 0 (Startkapital) einfügen, damit der Trichter spitz zuläuft
    x_axis = np.arange(1, days + 1)
    x_axis = np.insert(x_axis, 0, 0)
    median_path = np.insert(median_path, 0, start_capital)
    lower_bound = np.insert(lower_bound, 0, start_capital)
    upper_bound = np.insert(upper_bound, 0, start_capital)

    fig = go.Figure()

    # ---------------------------------------------------------
    # ETAGe 1: Der "Gefahren-Bereich" (Rot)
    # ---------------------------------------------------------
    # Wir zeichnen zuerst die untere rote Grenze
    fig.add_trace(go.Scatter(
        x=x_axis,
        y=lower_bound,
        mode='lines',
        line=dict(color='rgba(231, 76, 60, 0.8)', width=1), # Rot
        name=f'Worst Case ({lower_pct:g}% Quantil)'
    ))

    # Dann die Median-Linie. 'tonexty' füllt den Bereich hinunter zur roten Grenze.
    fig.add_trace(go.Scatter(
        x=x_axis,
        y=median_path,
        mode='lines',
        line=dict(color='rgb(52, 152, 219)', width=3), # Kräftiges Blau für den Median
        fill='tonexty', 
        fillcolor='rgba(231, 76, 60, 0.2)', # Transparentes Rot für die Füllung
        name='Median Pfad'
    ))

    # ---------------------------------------------------------
    # ETAGe 2: Der "Chancen-Bereich" (Grün)
    # ---------------------------------------------------------
    # Zum Schluss die obere grüne Grenze. 'tonexty' füllt hinunter zur Median-Linie.
    fig.add_trace(go.Scatter(
        x=x_axis,
        y=upper_bound,
        mode='lines',
        line=dict(color='rgba(46, 204, 113, 0.8)', width=1), # Grün
        fill='tonexty', 
        fillcolor='rgba(46, 204, 113, 0.2)', # Transparentes Grün für die Füllung
        name=f'Best Case ({upper_pct:g}% Quantil)'
    ))

    # Startkapital als Referenzlinie
    fig.add_hline(
        y=start_capital, 
        line_dash="dash", 
        line_color="rgba(255, 255, 255, 0.5)", 
        annotation_text="Startkapital"
    )

    # Layout optimieren
    fig.update_layout(
        title=title,
        xaxis_title="Handelstage",
        yaxis_title="Portfolio-Wert ($)",
        template="plotly_dark", 
        hovermode="x unified",
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

In [21]:

# Pfade fuer das Fan-Chart einmal explizit ziehen (calculate_monte_carlo_risk
# liefert sie als vierten Rueckgabewert; in Cell 27 werden sie mit _, _ verworfen).
_, _, _, pf_paths = calculate_monte_carlo_risk(
    portfolio_returns_log,
    100000,
    0.05,
    252,
    simulations=monte_carlo_simulations,
    black_swan=False,
)

fig = plot_monte_carlo_fan_chart(
    portfolio_paths=pf_paths,
    start_capital=100000,
    var_level=0.05,
    title="Lognormal: 1 Jahr",
)
fig.show()

In [22]:
def plot_historical_performance(portfolio_returns, market_returns, risk_free_returns, start_capital=100000, title="Historische Renditeentwicklung"):
    """
    Erstellt ein Liniendiagramm, das die historische Entwicklung von Portfolio, 
    Markt und risikofreiem Zins (auf Basis des Startkapitals) vergleicht.
    """
    # 1. Daten synchronisieren (wie bei den KPIs)
    aligned_data = pd.concat([
        portfolio_returns.squeeze().rename('Portfolio'), 
        market_returns.squeeze().rename('Market'), 
        risk_free_returns.squeeze().rename('RiskFree')
    ], axis=1, sort=False).dropna()

    # 2. Renditen in kumulatives Wachstum umrechnen (Zinseszinseffekt)
    # Formel: Startkapital * kumuliertes Produkt von (1 + Rendite)
    portfolio_growth = start_capital * (1 + aligned_data['Portfolio']).cumprod()
    market_growth = start_capital * (1 + aligned_data['Market']).cumprod()
    risk_free_growth = start_capital * (1 + aligned_data['RiskFree']).cumprod()

    # Wir fügen Tag 0 (Startkapital) ganz vorne an, damit alle Kurven exakt im selben Punkt starten
    portfolio_growth.loc[aligned_data.index[0] - pd.Timedelta(days=1)] = start_capital
    market_growth.loc[aligned_data.index[0] - pd.Timedelta(days=1)] = start_capital
    risk_free_growth.loc[aligned_data.index[0] - pd.Timedelta(days=1)] = start_capital
    
    # Sortieren, damit Tag 0 wirklich am Anfang steht
    portfolio_growth = portfolio_growth.sort_index()
    market_growth = market_growth.sort_index()
    risk_free_growth = risk_free_growth.sort_index()

    fig = go.Figure()

    # Portfolio Linie (Mag7)
    fig.add_trace(go.Scatter(
        x=portfolio_growth.index, y=portfolio_growth,
        mode='lines', line=dict(color='rgb(52, 152, 219)', width=2), # Blau
        name='MAG7 Portfolio'
    ))

    # Benchmark Linie (Markt)
    fig.add_trace(go.Scatter(
        x=market_growth.index, y=market_growth,
        mode='lines', line=dict(color='rgb(231, 76, 60)', width=2), # Rot
        name='Markt (Benchmark)'
    ))

    # Risk-Free Linie
    fig.add_trace(go.Scatter(
        x=risk_free_growth.index, y=risk_free_growth,
        mode='lines', line=dict(color='rgb(46, 204, 113)', width=2, dash='dot'), # Grün gepunktet
        name='Risk-Free Rate'
    ))

    # Layout optimieren
    fig.update_layout(
        title=title,
        xaxis_title="Datum",
        yaxis_title="Portfoliowert ($)",
        template="plotly_dark",
        hovermode="x unified",
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

In [23]:
# 1. Funktion aufrufen und mit deinen echten Daten füttern
fig_hist = plot_historical_performance(
    portfolio_returns = portfolio_returns_discrete, # oder portfolio_returns_log, je nachdem was du visualisieren willst
    market_returns = benchmark_world_data.pct_change().dropna(), 
    risk_free_returns = benchmark_risk_free_data.dropna() / 100 / 252, 
    start_capital = 100000,
    title = "MAG7 vs. Markt vs. Risikofreier Zins: Historische Entwicklung"
)

# 2. Grafik zeichnen lassen
fig_hist.show()

In [24]:
def plot_black_swan_comparison(paths_normal, paths_swan, start_capital=100000, title="Median-Vergleich: Normal vs. Black Swan"):
    """
    Legt die Median-Rendite der normalen Simulation und der Black-Swan-Simulation übereinander.
    """
    days = paths_normal.shape[0]
    
    # Mediane berechnen (quer über alle 10.000 Simulationen)
    median_normal = np.median(paths_normal, axis=1)
    median_swan = np.median(paths_swan, axis=1)
    
    # Tag 0 (Startkapital) einfügen
    x_axis = np.arange(1, days + 1)
    x_axis = np.insert(x_axis, 0, 0)
    median_normal = np.insert(median_normal, 0, start_capital)
    median_swan = np.insert(median_swan, 0, start_capital)
    
    fig = go.Figure()
    
    # Normale Linie (Blau)
    fig.add_trace(go.Scatter(
        x=x_axis, y=median_normal,
        mode='lines', line=dict(color='rgb(52, 152, 219)', width=3),
        name='Median (Ohne Crash)'
    ))
    
    # Black Swan Linie (Rot, gestrichelt für Dramatik)
    fig.add_trace(go.Scatter(
        x=x_axis, y=median_swan,
        mode='lines', line=dict(color='rgb(231, 76, 60)', width=3, dash='dash'),
        name='Median (Mit DotCom-Crash)'
    ))
    
    # Startkapital als Referenzlinie
    fig.add_hline(
        y=start_capital, 
        line_dash="dot", line_color="rgba(255, 255, 255, 0.3)", 
        annotation_text="Startkapital"
    )
    
    fig.update_layout(
        title=title, xaxis_title="Handelstage", yaxis_title="Portfolio-Wert ($)",
        template="plotly_dark", hovermode="x unified",
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    return fig

# Optionale Erweiterungen

## Sparplan, Inflation und Black-Swan-Szenario (optionaler Appendix)

In [25]:
def calculate_savings_plan_mc(log_returns, monthly_savings, years, simulations=10000,black_swan=False,inflation_rate=0.00):

    days = years * 252 #Anzahl der Handelstage über die gesamte Laufzeit
    clean_returns = log_returns.dropna()
    
    # Parameter für die Simulation
    mu = np.mean(clean_returns) #Erwartungswert der Renditen
    sigma = np.std(clean_returns)  #Volatilität der Renditen
    
    np.random.seed(monte_carlo_seed) # Für Reproduzierbarkeit
    
    daily_log_returns = np.random.normal(mu, sigma, (days, simulations)) #Generiert eine Matrix der täglichen Log-Renditen für alle Tage und Simulationen
    daily_simple_returns = np.exp(daily_log_returns) #Umwandlung der Log-Renditen
    
    #Hier entsteht die Black Swan Logik:

    if black_swan:
        crash_days = np.random.randint(0, days, size=simulations) #Wählt zufällig einen Tag für jeden Simulationspfad aus, an dem der Crash stattfindet
        
        for sim_idx in range(simulations): #In der Funktion wird sichergestellt, das jede Savings-Simulation später einen Crash-Tag hat
            crash_day = crash_days[sim_idx]
            daily_simple_returns[crash_day, sim_idx] *= 0.5 #Simuliert einen Crash an dem Portfoliowert sich halbiert


    #Hier entsteht die Sparplan-Logik:

    savings_matrix = np.zeros((days, simulations)) 
    for d in range(0, days, 21): #Annahme 252 Tage pro Jahr, also alle 21 Tage eine Einzahlung (252/12 = 21)
        savings_matrix[d, :] = monthly_savings #Matrix, die alle 21 Tage die Einzahlung enthält, ansonsten 0
    
    portfolio_values = np.zeros((days, simulations)) #Matrix, die die Portfoliowerte über die Zeit für alle Simulationen speichert
    current_values = np.zeros(simulations) #Startet bei 0, da wir mit einem Sparplan beginnen, kein Anfangskapital, speichert den aktuellen Portfoliowert von Tag zu Tag für jede Simulation
    
    for d in range(days):
        current_values = current_values * daily_simple_returns[d, :] + savings_matrix[d, :] # Rediten werden täglich aufsummiert, gleichzeitig werden alle 21 Tage eine Einzahlung hinzugefügt
        portfolio_values[d, :] = current_values #Speichert den Portfoliowert am Ende jedes Tages
        
    final_values = portfolio_values[-1, :] #Endwert der 10.000 Simulationen nach T Tagen
    
    #Inflationslogik:
    if inflation_rate > 0.0:
        discount_factor = (1 + inflation_rate) ** years # Abzinsung auf heutige Kaufkraft (Exponent sind die Jahre)
        final_values = final_values / discount_factor

    # VaR und ES berechnen (wie gehabt)
    # Aber Achtung: Hier vergleichen wir den Endwert mit der Summe der Einzahlungen!
    total_invested = (days // 21) * monthly_savings
    
    var_end_value = np.percentile(final_values, 5) # VAR-Schwellenwert finden (Das schlechteste 5 % der finalen Werte)
    es_end_value = np.mean(final_values[final_values <= var_end_value]) #AVG der Werte im 5% Tail
    
    # Profit/Loss im Vergleich zur Einzahlungssumme
    var_pnl = var_end_value - total_invested
    es_pnl = es_end_value - total_invested
    
    return var_pnl, es_pnl, final_values, total_invested

## Sparplanfunktion (optionaler Zusatzblock)

In [26]:
monatliche_rate = 100 
jahre_laufzeit = 20
inflation_rate = 0.02 #2% Inflation pro Jahr


# Funktionsaufruf
var_pnl, es_pnl, final_values, total_sum = calculate_savings_plan_mc(
    portfolio_returns_log, 
    monatliche_rate, 
    jahre_laufzeit, black_swan=True,inflation_rate=inflation_rate
)

# Schicke Formatierung der Ergebnisse
print(f"\n{' SPARPLAN ANALYSE (Monte Carlo) ':=^80}")
print(f"Zeitraum:            {jahre_laufzeit} Jahre")
print(f"Monatliche Rate:     {monatliche_rate:>10,.2f} $")
print(f"Gesamt investiert:   {total_sum:>10,.2f} $")
print("-" * 80)

# Interpretation der Ergebnisse
status_var = "GEWINN" if var_pnl > 0 else "VERLUST"
status_es = "GEWINN" if es_pnl > 0 else "VERLUST"

print(f"{'Value at Risk (95%):':<25} {abs(var_pnl):>12,.2f} $ {status_var}")
print(f"{'Expected Shortfall:':<25} {abs(es_pnl):>12,.2f} $ {status_es}")
print("-" * 80)
print(f"Durchschnittlicher Endwert: {np.mean(final_values):>12,.2f} $")
print("=" * 80)


======================== SPARPLAN ANALYSE (Monte Carlo) ========================
Zeitraum:            20 Jahre
Monatliche Rate:         100.00 $
Gesamt investiert:    24,000.00 $
--------------------------------------------------------------------------------
Value at Risk (95%):         55,426.06 $ GEWINN
Expected Shortfall:          33,622.74 $ GEWINN
--------------------------------------------------------------------------------
Durchschnittlicher Endwert:   693,997.78 $
